In [8]:
"""
Analyze p2f_vent_fio2 missing values, range, and ARDS classification in Supertables.

Root cause: MedTVT-R1's pickle raises ModuleNotFoundError(numpy._core.numeric), cannot load pkl.
Per-file subprocess fallback in notebook is too slow (~15 sec/file, 545k would take hundreds of hours).
Correct approach: run script in base env with multiprocessing, ~10–20 min. Data format verified OK.
"""
import subprocess

SCRIPT = "/work/gs285/Waveform_CXR_EHR/analyze_p2f_supertables.py"
print("Running analysis in base env (multiprocessing, ~10–20 min)...")
print("-" * 60, flush=True)

subprocess.run(
    ["conda", "run", "-n", "base", "python", SCRIPT],
    cwd="/work/gs285/Waveform_CXR_EHR",
    timeout=3600,
)

在 base 环境运行分析（多进程，约 10–20 分钟）...
------------------------------------------------------------
找到 545,888 个 supertable pkl 文件
使用 7 进程, 71 批, 每批约 7798 文件
  进度: 10/71 批, 当前有效值: 16,605
  进度: 20/71 批, 当前有效值: 33,890
  进度: 30/71 批, 当前有效值: 50,394
  进度: 40/71 批, 当前有效值: 67,869
  进度: 50/71 批, 当前有效值: 85,478
  进度: 60/71 批, 当前有效值: 101,990
  进度: 70/71 批, 当前有效值: 116,629
  进度: 71/71 批, 当前有效值: 118,438

[p2f_vent_fio2 统计]
总行数: 64,846,574
有值: 118,438
缺失: 64,728,136
缺失比例: 99.82%

数值范围: min=25.00, max=2104.76, mean=243.49 (median 略)

按 ARDS 严重程度 (PaO2/FiO2):
  Mild (200 < P/F ≤ 300):      30229   条
  Moderate (100 < P/F ≤ 200):  39539   条
  Severe (P/F ≤ 100):          16621   条
  Above Mild (P/F > 300):      32049   条



CompletedProcess(args=['conda', 'run', '-n', 'base', 'python', '/work/gs285/Waveform_CXR_EHR/analyze_p2f_supertables.py'], returncode=0)

In [7]:
# Quick verify: load single pkl structure in base env (optional)
import subprocess
out = subprocess.run(
    ["conda", "run", "-n", "base", "python", "-c", '''
import pickle
from pathlib import Path
p = Path("/hpc/group/kamaleswaranlab/mimic_iv/sepy_output/mimic-supertables/Supertables/20000147.pkl")
with open(p, "rb") as f:
    st = pickle.load(f)
print("type:", type(st).__name__, "shape:", st.shape)
print("p2f_vent_fio2 in columns:", "p2f_vent_fio2" in st.columns)
s = st["p2f_vent_fio2"]
print("non-null count:", s.notna().sum(), "sample:", s.dropna().head(3).tolist())
'''],
    capture_output=True, text=True, timeout=30
)
print(out.stdout or out.stderr)

type: DataFrame shape: (96, 165)
p2f_vent_fio2 in columns: True
非空数: 3 样例: [506.0, 236.0, 208.0]




In [10]:
# List ALL columns in Supertables (load sample pkl in base env)
import subprocess
out = subprocess.run(
    ["conda", "run", "-n", "base", "python", "-c", '''
import pickle
from pathlib import Path

pkl_dir = Path("/hpc/group/kamaleswaranlab/mimic_iv/sepy_output/mimic-supertables/Supertables")
# Use a known sample file (avoid sorting 545k files)
p = pkl_dir / "20000019.pkl"
with open(p, "rb") as f:
    st = pickle.load(f)

cols = list(st.columns)
print(f"Sample file: {p.name}")
print(f"Supertables: shape {st.shape} (rows=timepoints, cols=variables)")
print(f"Total columns: {len(cols)}")
print()
print("All column names:")
for i, c in enumerate(cols):
    print(f"  {i+1:3d}. {c}")
'''],
    capture_output=True, text=True, timeout=30
)
print(out.stdout or out.stderr)

Sample file: 20000019.pkl
Supertables: shape (70, 173) (rows=timepoints, cols=variables)
Total columns: 173

All column names:
    1. age
    2. gender
    3. race
    4. cci9
    5. cci10
    6. anion_gap
    7. base_excess
    8. bicarb_(hco3)
    9. blood_urea_nitrogen_(bun)
   10. calcium
   11. calcium_ionized
   12. chloride
   13. creatinine
   14. gfr
   15. glucose
   16. magnesium
   17. osmolarity
   18. phosphorus
   19. potassium
   20. sodium
   21. haptoglobin
   22. hematocrit
   23. hemoglobin
   24. met_hgb
   25. platelets
   26. white_blood_cell_count
   27. carboxy_hgb
   28. alanine_aminotransferase_(alt)
   29. albumin
   30. alkaline_phosphatase
   31. ammonia
   32. aspartate_aminotransferase_(ast)
   33. bilirubin_direct
   34. bilirubin_total
   35. fibrinogen
   36. inr
   37. lactate_dehydrogenase
   38. lactic_acid
   39. partial_prothrombin_time_(ptt)
   40. prealbumin
   41. protein
   42. prothrombin_time_(pt)
   43. thrombin_time
   44. transferrin
   

In [ ]:
# Extract all rows where p2f_vent_fio2 has a value; 顺序处理，找到即存 (~118k rows, 可能 1–2 小时)
import subprocess

EXTRACT_SCRIPT = "/work/gs285/Waveform_CXR_EHR/extract_p2f_rows.py"
OUTPUT_CSV = "/work/gs285/Waveform_CXR_EHR/p2f_vent_fio2_valid_rows.csv"

print("Extracting rows with valid p2f_vent_fio2 to CSV...")
print("Output:", OUTPUT_CSV)
print("-" * 60, flush=True)

subprocess.run(
    ["conda", "run", "-n", "base", "python", EXTRACT_SCRIPT],
    cwd="/work/gs285/Waveform_CXR_EHR",
    timeout=7200,
)

Extracting rows with valid p2f_vent_fio2 to CSV...
Output: /work/gs285/Waveform_CXR_EHR/p2f_vent_fio2_valid_rows.csv
------------------------------------------------------------


In [1]:
# Check p2f_vent_fio2_enriched.csv: rows where both CXR_signal and ECG_signal are 1
import pandas as pd

df = pd.read_csv("/work/gs285/Waveform_CXR_EHR/p2f_vent_fio2_enriched.csv", low_memory=False)
both = ((df["CXR_signal"] == 1) & (df["ECG_signal"] == 1)).sum()
cxr_only = ((df["CXR_signal"] == 1) & (df["ECG_signal"] != 1)).sum()
ecg_only = ((df["CXR_signal"] != 1) & (df["ECG_signal"] == 1)).sum()

print("Rows where both CXR_signal=1 AND ECG_signal=1:", both)
print()
print("Summary:")
print(f"  Total rows:           {len(df):,}")
print(f"  CXR_signal=1:         {(df['CXR_signal']==1).sum():,}")
print(f"  ECG_signal=1:         {(df['ECG_signal']==1).sum():,}")
print(f"  Both CXR & ECG:       {both:,}")
print(f"  CXR only (no ECG):    {cxr_only:,}")
print(f"  ECG only (no CXR):    {ecg_only:,}")

Rows where both CXR_signal=1 AND ECG_signal=1: 2288

Summary:
  Total rows:           118,438
  CXR_signal=1:         9,830
  ECG_signal=1:         13,946
  Both CXR & ECG:       2,288
  CXR only (no ECG):    7,542
  ECG only (no CXR):    11,658
